In [ ]:
import os

from core import *
from processing import *
from utils import *

In [ ]:
KEY = "LuxPest_DB" #"40CFR302" #"ALGALTOX"

class SMILES_Data(BaseModel):  # Phase of the molecule
    SMILES_columns: List[str] = Field(default_factory=list, description="Columns in csv file that contain SMILES")

class SMILESandCSV_Data(SMILES_Data):
    CSV_file_name: str = Field(description="Name of the CSV file.")


class EssentialColumnAnalysis(BaseModel):
    file_and_columns: List[SMILESandCSV_Data]

d,_ = VersionedYaml(Path(WORKSPACE) / KEY).read("ess_columns"); print(d)

typed_d = EssentialColumnAnalysis.model_validate(d)   # pydantic v2
typed_d


In [ ]:
import pandas as pd


class SmilesRecordsIterable:
    def __init__(self, workspace: str, dataset_key: str, typed_cfg: EssentialColumnAnalysis):
        self.workspace = workspace
        self.dataset_key = dataset_key
        self.typed_cfg = typed_cfg

    def __iter__(self):
        for file_cfg in self.typed_cfg.file_and_columns:
            file_key = file_cfg.CSV_file_name
            path = Path(self.workspace) / self.dataset_key / file_key
            df = pd.read_csv(path)

            for column_key in file_cfg.SMILES_columns:
                for raw_smiles in df[column_key]:
                    for substructure_key, sub_smiles in enumerate(split_smiles(raw_smiles)):
                        yield {
                            "SMILES": sub_smiles,
                            "dataset_key": self.dataset_key,
                            "file_key": file_key,
                            "column_key": column_key,
                            "substructure_key": substructure_key,
                        }


records = SmilesRecordsIterable(WORKSPACE, KEY, typed_d)
it = iter(records)
[next(it) for i in range(10)]

In [ ]:
from processing import BasicAnalysis

os.makedirs(Path(WORKSPACE_DATA)/"40CFR302/", exist_ok=True)

analysis = BasicAnalysis(
    dataset_key="40CFR302",
    file_in= Path(WORKSPACE)/ "40CFR302/chemicals.csv",  # input CSV
    smiles_columns_list=["smiles"],            # adjust to your real column names
    file_full_desc=Path(WORKSPACE_DATA) / "40CFR302" / "full_smiles.csv",
    file_unique_desc=Path(WORKSPACE_DATA) / "40CFR302" / "unique_smiles.csv",
    file_broken_desc=Path(WORKSPACE_DATA) / "40CFR302" / "broken_smiles.csv",
    file_dataset_summary=Path(WORKSPACE_DATA) / "40CFR302" / "dataset_summary.json",
    chunk_size=50000,
)

analysis.process()

In [ ]:
!ls -l ../workspace_DATA/

In [ ]:
class BasicAnalysis2:
    """Notebook-local version of BasicAnalysis using SmilesRecordsIterable."""

    def __init__(self,
                 records,
                 dataset_key: str,
                 file_full_desc: str,
                 file_unique_desc: str,
                 file_broken_desc: str,
                 file_dataset_summary: str,
                 flush_every: int = 50000) -> None:
        self.records = records
        self.dataset_key = dataset_key
        self.file_full_desc = file_full_desc
        self.file_unique_desc = file_unique_desc
        self.file_broken_desc = file_broken_desc
        self.file_dataset_summary = file_dataset_summary
        self.flush_every = flush_every

        self.processor = ParameterProcessor()
        self.aggregation = AggregationProcessor()

    def process(self):
        GLOBAL_DICT_OF_UNIQUE = {None: {None: 0}}  # InChIKey -> {SMILES: count}
        GLOBAL_STATISTICS_FULL = {
            "dataset_key": self.dataset_key,
            "number_of_records": 0,
            "number_of_molecules": 0,
            "number_of_unique_molecules": 0,
            "number_of_SMILES_with_error": 0,
            "number_of_SMILES_with_warning": 0,
        }

        list_dict_broken = []
        list_dict_full = []
        list_dict_unique = []

        _header_written_broken_SMILES = False
        _header_written_full_SMILES = False
        _header_written_unique_SMILES = False

        def flush_buffers():
            nonlocal _header_written_broken_SMILES
            nonlocal _header_written_full_SMILES
            nonlocal _header_written_unique_SMILES

            if len(list_dict_broken) > 0:
                df_processed = pd.DataFrame(list_dict_broken)
                df_processed.to_csv(
                    self.file_broken_desc,
                    mode="a",
                    index=False,
                    header=not _header_written_broken_SMILES,
                )
                _header_written_broken_SMILES = True
                list_dict_broken.clear()

            if len(list_dict_unique) > 0:
                df_processed = pd.DataFrame(list_dict_unique)
                df_processed.to_csv(
                    self.file_unique_desc,
                    mode="a",
                    index=False,
                    header=not _header_written_unique_SMILES,
                )
                _header_written_unique_SMILES = True
                list_dict_unique.clear()

            if len(list_dict_full) > 0:
                df_processed = pd.DataFrame(list_dict_full)
                df_processed.to_csv(
                    self.file_full_desc,
                    mode="a",
                    index=False,
                    header=not _header_written_full_SMILES,
                )
                _header_written_full_SMILES = True
                list_dict_full.clear()

        Path(self.file_full_desc).parent.mkdir(parents=True, exist_ok=True)
        Path(self.file_unique_desc).parent.mkdir(parents=True, exist_ok=True)
        Path(self.file_broken_desc).parent.mkdir(parents=True, exist_ok=True)
        Path(self.file_dataset_summary).parent.mkdir(parents=True, exist_ok=True)

        for i, record in enumerate(self.records, start=1):
            GLOBAL_STATISTICS_FULL["number_of_records"] += 1

            smiles = record["SMILES"]
            norm_SMILES, InChIKey, is_warn, msg, mol_h = normalize_smiles_and_inchikey(smiles)

            meta = {
                "dataset_key": record["dataset_key"],
                "file_key": record["file_key"],
                "column_key": record["column_key"],
                "substructure_key": record["substructure_key"],
            }

            if (norm_SMILES is None) or is_warn:
                list_dict_broken.append({
                    **meta,
                    "SMILES": smiles,
                    "norm_SMILES": norm_SMILES,
                    "is_broken": (norm_SMILES is None),
                    "is_warning": is_warn,
                    "msg": msg,
                })
                if norm_SMILES is None:
                    GLOBAL_STATISTICS_FULL["number_of_SMILES_with_error"] += 1
                else:
                    GLOBAL_STATISTICS_FULL["number_of_SMILES_with_warning"] += 1

            if norm_SMILES is not None:
                dict_full = {
                    **meta,
                    "norm_SMILES": norm_SMILES,
                    "InChIKey": InChIKey,
                }
                dict_full = self.processor.process(norm_SMILES, mol_h, record, dict_full)
                list_dict_full.append(dict_full)
                GLOBAL_STATISTICS_FULL["number_of_molecules"] += 1

                is_store_unique = True
                if InChIKey in GLOBAL_DICT_OF_UNIQUE:
                    is_store_unique = not (norm_SMILES in GLOBAL_DICT_OF_UNIQUE[InChIKey])

                if is_store_unique:
                    list_dict_unique.append(dict_full)
                    GLOBAL_STATISTICS_FULL["number_of_unique_molecules"] += 1
                    GLOBAL_STATISTICS_FULL = self.aggregation.process(dict_full, GLOBAL_STATISTICS_FULL)

                    if InChIKey not in GLOBAL_DICT_OF_UNIQUE:
                        GLOBAL_DICT_OF_UNIQUE[InChIKey] = {norm_SMILES: 0}
                    else:
                        if norm_SMILES not in GLOBAL_DICT_OF_UNIQUE[InChIKey]:
                            GLOBAL_DICT_OF_UNIQUE[InChIKey][norm_SMILES] = 0
                        else:
                            GLOBAL_DICT_OF_UNIQUE[InChIKey][norm_SMILES] += 1

            if i % self.flush_every == 0:
                flush_buffers()

        flush_buffers()

        GLOBAL_STATISTICS_FULL = self.aggregation.finalize(GLOBAL_STATISTICS_FULL)
        with open(self.file_dataset_summary, "w") as json_file:
            json.dump(
                GLOBAL_STATISTICS_FULL,
                json_file,
                default=lambda o: list(o) if isinstance(o, set) else o,
                indent=2,
            )


records = SmilesRecordsIterable(WORKSPACE, KEY, typed_d)
analysis2 = BasicAnalysis2(
    records=records,
    dataset_key=KEY,
    file_full_desc=f"{WORKSPACE_DATA}/{KEY}/full_smiles2.csv",
    file_unique_desc=f"{WORKSPACE_DATA}/{KEY}/unique_smiles2.csv",
    file_broken_desc=f"{WORKSPACE_DATA}/{KEY}/broken_smiles2.csv",
    file_dataset_summary=f"{WORKSPACE_DATA}/{KEY}/dataset_summary2.json",
)

analysis2.process()

In [ ]:
# Run full SMILES processing for all keys
all_keys = sorted(
    p.name
    for p in Path(WORKSPACE).iterdir()
    if p.is_dir() and not p.name.startswith(".")
)

all_keys

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp



def _run_one_key(key: str):
    d, _ = VersionedYaml(Path(WORKSPACE) / key).read("ess_columns")
    typed_cfg = EssentialColumnAnalysis.model_validate(d)

    records = SmilesRecordsIterable(WORKSPACE, key, typed_cfg)
    out_dir = Path(WORKSPACE_DATA) / key
    out_dir.mkdir(parents=True, exist_ok=True)

    analysis2 = BasicAnalysis2(
        records=records,
        dataset_key=key,
        file_full_desc=str(out_dir / "full_smiles2.csv"),
        file_unique_desc=str(out_dir / "unique_smiles2.csv"),
        file_broken_desc=str(out_dir / "broken_smiles2.csv"),
        file_dataset_summary=str(out_dir / "dataset_summary2.json"),
    )
    analysis2.process()
    return {"key": key, "status": "ok"}



def get_real_core_count() -> int:
    # Prefer physical cores (real cores). Fallback to logical if unavailable.
    try:
        import psutil
        n = psutil.cpu_count(logical=False)
        if n is not None and n > 0:
            return n
    except Exception:
        pass

    n = os.cpu_count()
    return n if n is not None and n > 0 else 1



def run_all_keys_parallel(all_keys, max_workers=None):
    # Linux notebook: use fork so notebook-defined classes/functions work in workers.
    ctx = mp.get_context("fork")

    if max_workers is None:
        max_workers = get_real_core_count()

    results = []
    with ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(_run_one_key, key): key for key in all_keys}
        for future in as_completed(futures):
            key = futures[future]
            try:
                results.append(future.result())
                print(f"OK: {key}")
            except Exception as e:
                results.append({"key": key, "status": "error", "error": str(e)})
                print(f"ERROR: {key} -> {e}")

    return pd.DataFrame(results).sort_values("key").reset_index(drop=True)


# Example
result_df = run_all_keys_parallel(all_keys)  # uses number of real cores
result_df